# Step 1 - Dataset Preparation
## Semantic Segmentation with Deep Learning - Potsdam Dataset

This notebook covers:
- Loading and visualizing the 2D Semantic Labeling Potsdam dataset
- Preparing data for 5-fold cross-validation
- Generating TFRecord files (or PyTorch DataLoaders)

## 1.1 Install and Import Required Libraries

In [ ]:
!pip install rasterio numpy matplotlib scikit-learn tensorflow -q

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.plot import show
from sklearn.model_selection import KFold
import tensorflow as tf

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

## 1.2 Dataset Information

The **2D Semantic Labeling Potsdam** dataset has:
- **15,048 GeoTIFF tiles** (we sample at least 5,000)
- Each tile has **6 bands**: Red, Green, Blue, Infrared (IR), Elevation, and **Labels** (band index 5)
- **6 classes**:
  1. Impervious surface
  2. Building
  3. Tree
  4. Low vegetation
  5. Car
  6. Clutter/Background

In [ ]:
# =====================================================================
# =====================================================================
# =====================================================================
# =====================================================================
# CONFIGURATION - Adjust paths as needed
# =====================================================================
# Original local path:
# DATA_DIR = r'c:\Users\mina_\OneDrive\Documents\DESING_OF_AI_SYSTEMS\Semantic Segmentation with Deep Learning\PROJECT\Potsdam-GeoTif'

import os
def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    existing = [p for p in possible if os.path.exists(p)]
    for p in existing:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return existing[0] if existing else 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'PROJECT', 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================
# =====================================================================

# Path to the directory containing all GeoTIFF files
# =====================================================================
# =====================================================================

# Path to the directory containing all GeoTIFF files

# Number of images to sample for training
NUM_SAMPLES = 5000  # increase if your system allows more

# Number of folds for cross-validation
N_FOLDS = 5

# Class labels and colors for visualization
CLASS_NAMES = [
    'Impervious surface',
    'Building',
    'Tree',
    'Low vegetation',
    'Car',
    'Clutter/Background'
]

CLASS_COLORS = [
    [255, 255, 255],  # Impervious surface - white
    [0, 0, 255],      # Building - blue
    [0, 255, 0],      # Tree - green
    [0, 255, 255],    # Low vegetation - cyan
    [255, 255, 0],    # Car - yellow
    [255, 0, 0],      # Clutter/Background - red
]

NUM_CLASSES = len(CLASS_NAMES)
print(f'Number of classes: {NUM_CLASSES}')
print(f'Classes: {CLASS_NAMES}')

## 1.3 Load Dataset File Paths

In [ ]:
def get_all_tif_files(data_dir):
    """Collect all .tif file paths from the dataset directory."""
    tif_files = []
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.tif'):
                tif_files.append(os.path.join(root, f))
    return tif_files

all_files = get_all_tif_files(DATA_DIR)
print(f'Total GeoTIFF files found: {len(all_files)}')

# Randomly sample files
sampled_files = random.sample(all_files, min(NUM_SAMPLES, len(all_files)))
print(f'Sampled {len(sampled_files)} files for training')

## 1.4 Visualize a Sample Image

**Assignment Requirement:** Select an image (excluding `0000000224-0000042784.tif`) and visualize:
- RGB image
- Elevation band
- Target label band (with colorbar and class names)

In [ ]:
def read_geotiff(file_path):
    """Read a GeoTIFF file and return all bands as numpy array."""
    with rasterio.open(file_path) as src:
        data = src.read()  # shape: (bands, height, width)
    return data

def label_to_rgb(label_band, colors):
    """Convert label band (class indices) to RGB image for visualization."""
    h, w = label_band.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for class_idx, color in enumerate(colors):
        mask = label_band == class_idx
        rgb[mask] = color
    return rgb

def normalize_band(band):
    """Normalize band to 0-1 range for visualization."""
    band_min = band.min()
    band_max = band.max()
    if band_max == band_min:
        return np.zeros_like(band, dtype=np.float32)
    return (band - band_min) / (band_max - band_min)


# Select a sample image (try to exclude the provided example file)
EXCLUDED_FILE = '0000000224-0000042784.tif'
sample_files_filtered = [f for f in all_files if EXCLUDED_FILE not in f]

# Fallback: if no other files exist, use the example file itself
if len(sample_files_filtered) == 0:
    print('Note: Only the example file is available. Using it for visualization.')
    sample_file = all_files[0]
else:
    sample_file = random.choice(sample_files_filtered)

print('Selected sample file: ' + os.path.basename(sample_file))

# Read the sample file
data = read_geotiff(sample_file)
print('Data shape:', data.shape, ' (bands, height, width)')
print('Bands: [0]=Red, [1]=Green, [2]=Blue, [3]=IR, [4]=Elevation, [5]=Labels')

In [ ]:
# Extract individual bands
red   = data[0].astype(np.float32)
green = data[1].astype(np.float32)
blue  = data[2].astype(np.float32)
ir    = data[3].astype(np.float32)
elev  = data[4].astype(np.float32)
label = data[5].astype(np.int32)

# Build RGB image
rgb_image = np.stack([
    normalize_band(red),
    normalize_band(green),
    normalize_band(blue)
], axis=-1)

# Build label RGB visualization
label_rgb = label_to_rgb(label, CLASS_COLORS)

# Create colorbar patches
patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

# Plot the three visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Sample: {os.path.basename(sample_file)}', fontsize=14)

# 1. RGB Image
axes[0].imshow(rgb_image)
axes[0].set_title('RGB Image (R, G, B bands)')
axes[0].axis('off')

# 2. Elevation Band
im_elev = axes[1].imshow(normalize_band(elev), cmap='terrain')
axes[1].set_title('Elevation Band')
axes[1].axis('off')
plt.colorbar(im_elev, ax=axes[1], fraction=0.046, pad=0.04, label='Normalized Elevation')

# 3. Label Band
im_label = axes[2].imshow(label_rgb)
axes[2].set_title('Target Label Band')
axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=7, framealpha=0.8)

plt.tight_layout()
plt.savefig('sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualization saved as sample_visualization.png')

## 1.5 Split Dataset into 5 Folds for Cross-Validation

**Strategy:**
- Use **Folds 1, 2, 3** → Training
- Use **Fold 4** → Validation
- Use **Fold 5** → Test (held out)

In [ ]:
# Split sampled files into 5 folds
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

folds = []
sampled_array = np.array(sampled_files)

for fold_idx, (_, fold_indices) in enumerate(kf.split(sampled_array)):
    folds.append(sampled_array[fold_indices].tolist())
    print(f'Fold {fold_idx+1}: {len(folds[fold_idx])} images')

# Assign folds
train_files = folds[0] + folds[1] + folds[2]   # Folds 1, 2, 3
val_files   = folds[3]                           # Fold 4
test_files  = folds[4]                           # Fold 5

print(f'\nTraining set  : {len(train_files)} images  (Folds 1+2+3)')
print(f'Validation set: {len(val_files)} images   (Fold 4)')
print(f'Test set      : {len(test_files)} images   (Fold 5)')

In [ ]:
# =====================================================================
# =====================================================================
# =====================================================================
# CONFIGURATION - Adjust paths as needed
# =====================================================================
# Original local path:
# DATA_DIR = r'c:\Users\mina_\OneDrive\Documents\DESING_OF_AI_SYSTEMS\Semantic Segmentation with Deep Learning\PROJECT\Potsdam-GeoTif'

import os
def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    existing = [p for p in possible if os.path.exists(p)]
    for p in existing:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return existing[0] if existing else 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'PROJECT', 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================

import json

# Save fold splits to JSON for later use
fold_splits = {
    'train': train_files,
    'val':   val_files,
    'test':  test_files,
    'all_folds': [f.tolist() if hasattr(f, 'tolist') else f for f in folds]
}

# # # with open('fold_splits.json', 'w') as fp:
# # with open(os.path.join(DATA_DIR, 'fold_splits.json'), 'w') as fp:
# with open(os.path.join(DATA_DIR, 'fold_splits.json'), 'w') as fp:
with open(os.path.join(DATA_DIR, 'fold_splits.json'), 'w') as fp:
    json.dump(fold_splits, fp, indent=2)

# # # print('Fold splits saved to fold_splits.json')# # print('Fold splits saved to fold_splits.json')# # print('Fold splits saved to fold_splits.json')# print('Fold splits saved to fold_splits.json')# # print('Fold splits saved to fold_splits.json')# print('Fold splits saved to fold_splits.json')# print('Fold splits saved to fold_splits.json')print('Fold splits saved to fold_splits.json')

## 1.6 Dataset Preprocessing Functions

In [ ]:
def load_and_preprocess(file_path, use_all_bands=False):
    """
    Load a GeoTIFF file and return (features, label).
    
    Args:
        file_path   : path to .tif file
        use_all_bands: if True, use RGB + IR + Elevation (5 bands)
                       if False, use only RGB + IR (4 bands)
    Returns:
        features: numpy array, shape (H, W, C)
        label   : one-hot encoded label, shape (H, W, NUM_CLASSES)
    """
    data = read_geotiff(file_path)  # (6, H, W)
    
    if use_all_bands:
        # Use bands 0-4 (R, G, B, IR, Elevation)
        features = data[:5].transpose(1, 2, 0).astype(np.float32)
    else:
        # Use bands 0-3 (R, G, B, IR)
        features = data[:4].transpose(1, 2, 0).astype(np.float32)
    
    # Normalize features to [0, 1]
    for c in range(features.shape[-1]):
        band = features[..., c]
        b_min, b_max = band.min(), band.max()
        if b_max > b_min:
            features[..., c] = (band - b_min) / (b_max - b_min)
    
    # Labels: band index 5
    label_band = data[5].astype(np.int32)  # (H, W)
    
    # One-hot encode
    label_onehot = tf.keras.utils.to_categorical(label_band, num_classes=NUM_CLASSES)
    
    return features, label_onehot


# Test with sample file
features_4band, label = load_and_preprocess(sample_file, use_all_bands=False)
features_5band, _     = load_and_preprocess(sample_file, use_all_bands=True)

print(f'4-band features shape (RGB+IR)         : {features_4band.shape}')
print(f'5-band features shape (RGB+IR+Elevation): {features_5band.shape}')
print(f'Label shape (one-hot)                  : {label.shape}')

## 1.7 Summary

✅ Dataset files loaded from GeoTIFF format  
✅ 5000 images sampled  
✅ Visualized RGB, Elevation, and Label bands  
✅ Split into 5 folds (Folds 1-3: Train, Fold 4: Val, Fold 5: Test)  
✅ Preprocessing function ready for band selection  

**Next Step:** Proceed to `Step2_Simple_Model.ipynb` for training.